In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import pandas as pd

df = pd.read_csv("yorumlar.csv")
print(f"{len(df)} yorum yüklendi, kolonlar: kolonlar: {list(df.columns)}")


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class YorumAnaliz(BaseModel):
    sentiment: Literal["pozitif","negatif","nötr"] = Field(description= "Genel duygu")
    konu : Literal["kargo","fiyat","kalite", "diger"] = Field(description= "Ana Konu")
    aciliyet: int = Field(ge= 1, le= 5, description= "1=önemsiz ... 5 = acil müdahale")

In [ ]:
from langchain_openai import ChatOpenAI

def llm_ile_etiketle(metin:str) -> dict:
    llm = ChatOpenAI(model= "gpt-5.4-mini", temperature= 0).with_structured_output(YorumAnaliz)
    return llm.invoke(f"Şu müşteri yorumunu analiz et: \n{metin}").model_dump()

sonuc = llm_ile_etiketle(df["yorum"][0])
print("Canlı LLM Çıktısı:", sonuc)

In [ ]:
df_etiketli = pd.read_csv("yorumlar_etiketli.csv")
print(f"{len(df_etiketli)} yorum etiketlendi. Artık yapısal veri elimizde:")
df_etiketli.head()

In [ ]:
kayitlar = [llm_ile_etiketle(m) for m in df["yorum"]]
df_etiketli = pd.concat([df, pd.DataFrame(kayitlar)], axis= 1)
df_etiketli.to_csv("yorumlar_etiketli.csv", index = False)

df_etiketli = pd.read_csv("yorumlar_etiketli.csv")
print(f"{len(df_etiketli)} yorum etiketlendi. Artık yapısal veri elimizde")
df_etiketli.head()

# İÇGÖRÜ: Etiketlerden İş Değeri Üret (API'siz çalışır)

İşin asıl amacı buydu. Artık elimizde yapısal veri var; sıradan pandas ile yöneticinin sorusuna cevap veriyoruz: "Nerede sıkıntı var?"

In [2]:
# 1) En çok hangi konuda yorum geliyor?
print("Konu Dağılımı")
print(df_etiketli["konu"].value_counts(), "\n")

# 2) ŞİKAYETLER (negatif) hangi konuda yoğun? -> asıl sorun nerede?
neg = df_etiketli[df_etiketli["sentiment"] == "negatif"]
print(f"NEGATİF YORUMLARIN KONU DAGILIMI (toplam {len(neg)} şikayet)")
print(neg["konu"].value_counts(), "\n")

# 3) Hangi konu en 'acil'? (ortalama aciliyet)
print("KONU BAŞINA ORTALAMA ACİLİYET")
print(df_etiketli.groupby("konu")["aciliyet"].mean().round(2).sort_values(ascending= False))

Konu Dağılımı


NameError: name 'df_etiketli' is not defined

In [ ]:
# 4) Aksiyon Listesi: en acil 3 şikayet (operasyon ekibine git der)

print("EN ACİL 3 ŞİKAYET:")
for _ , r in neg.nlargest(3, "aciliyet").iterrows():
    print(f" [aciliyet {r["aciliyet"]}] {r["konu"]}")

# DOĞRULA: LLM'e Güvenelim mi? (API'siz çalışır)


In [ ]:
altin_kume = pd.read_csv("yorumlar_etiketli_new.csv").head(12).rename(columns= {"sentiment": "llm_tahmin"})
altin_kume["insan_etiket"] = [
    "pozitif", "negatif","notr","negatif","pozitif","notr",
    "pozitif","negatif","pozitif","negatif","notr","negatif"
]

altin_kume[["yorum","insan_etiket","llm_tahmin"]]

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true = altin_kume["insan_etiket"]
y_pred = altin_kume["llm_tahmin"]

print(f"Accuracy (dogruluk): {accuracy_score(y_true, y_pred)}")
print(f"\n12 yorumdan 10'unda LLM insanla aynı etiketi verdi -> 2 hata. Bunu ÖLÇTÜK, varsaymadık.")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

etiketler = ["pozitif","negatif","nötr"]
print("Confusion matrix (satır = altın/dogru, sütun = LLM):")
print(pd.DataFrame(confusion_matrix(y_true,y_pred, labels= etiketler),
                   index = etiketler, columns = etiketler))
print()
print(classification_report(y_true, y_pred, labels= etiketler, zero_division= 0))
